In [ ]:
# ============================================================
# CELL 1 — MODEL 2 LIGHTGBM INFERENCE SETUP
# ============================================================

import os
import json
import numpy as np

from sentence_transformers import SentenceTransformer
import lightgbm as lgb

print("=" * 60)
print("MODEL 2 — LIGHTGBM INFERENCE")
print("=" * 60)

# ------------------------------------------------------------
# Base directory
# ------------------------------------------------------------

BASE_DIR = os.getcwd()

# ------------------------------------------------------------
# Preprocessed files
# ------------------------------------------------------------

PREPROCESSED_DIR = os.path.join(
    BASE_DIR,
    "model2_preprocessed_lightgbm"
)

LABEL_MAPPING_PATH = os.path.join(
    PREPROCESSED_DIR,
    "label_mapping.json"
)

# ------------------------------------------------------------
# Trained LightGBM model
# ------------------------------------------------------------

MODEL_PATH = os.path.join(
    BASE_DIR,
    "outputs",
    "model2_lightgbm",
    "final",
    "model.txt"
)

# ------------------------------------------------------------
# Embedding model
# ------------------------------------------------------------

EMBEDDING_MODEL_NAME = "all-mpnet-base-v2"

print("\nBase directory:")
print(BASE_DIR)

print("\nLightGBM model:")
print(MODEL_PATH)

print("\nLabel mapping:")
print(LABEL_MAPPING_PATH)

print("\nEmbedding model:")
print(EMBEDDING_MODEL_NAME)

c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0910 21:53:30.320000 31596 Lib\site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


MODEL 2 — LIGHTGBM INFERENCE

Base directory:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)

LightGBM model:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\outputs\model2_lightgbm\final\model.txt

Label mapping:
c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm\label_mapping.json

Embedding model:
all-mpnet-base-v2


In [2]:
# ============================================================
# CELL 2 — LOAD SAVED MODEL COMPONENTS
# ============================================================

# ------------------------------------------------------------
# Load label mapping
# ------------------------------------------------------------

with open(
    LABEL_MAPPING_PATH,
    "r",
    encoding="utf-8"
) as f:
    label_mapping = json.load(f)

label2id = label_mapping["label2id"]
id2label = label_mapping["id2label"]

# ------------------------------------------------------------
# Load trained LightGBM model
# ------------------------------------------------------------

model = lgb.Booster(
    model_file=MODEL_PATH
)

# ------------------------------------------------------------
# Load SentenceTransformer
# ------------------------------------------------------------

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

# ------------------------------------------------------------
# Display verification
# ------------------------------------------------------------

print("=" * 60)
print("INFERENCE COMPONENTS LOADED")
print("=" * 60)

print("\nLightGBM model loaded: YES")

print("Number of classes:")
print(model.num_model_per_iteration())

print("\nLabel mappings:")
print(len(label2id))

print("\nEmbedding model:")
print(EMBEDDING_MODEL_NAME)

print("\nInference setup: READY")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 11287.97it/s]


INFERENCE COMPONENTS LOADED

LightGBM model loaded: YES
Number of classes:
36

Label mappings:
36

Embedding model:
all-mpnet-base-v2

Inference setup: READY


In [3]:
# ============================================================
# CELL 3 — CREATE INFERENCE FUNCTION
# ============================================================

def flatten_context(context):
    """
    Convert structured dialogue into the same text format
    used during training.
    """

    messages = []

    for message in context:
        role = message["role"].strip().lower()
        content = message["content"].strip()

        if role == "user":
            messages.append(f"User: {content}")

        elif role == "assistant":
            messages.append(f"Assistant: {content}")

        else:
            messages.append(
                f"{role.capitalize()}: {content}"
            )

    return " | ".join(messages)


def predict_next_slot(context, top_k=3):
    """
    Predict the next dialogue-management label.

    Parameters
    ----------
    context : list
        Conversation messages in the format:
        [
            {"role": "user", "content": "..."},
            {"role": "assistant", "content": "..."}
        ]

    top_k : int
        Number of top predictions to return.

    Returns
    -------
    dict
        Predicted label, confidence, and top-k predictions.
    """

    # --------------------------------------------------------
    # Flatten conversation
    # --------------------------------------------------------

    text = flatten_context(context)

    # --------------------------------------------------------
    # Generate embedding
    # --------------------------------------------------------

    embedding = embedding_model.encode(
        [text],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # --------------------------------------------------------
    # LightGBM prediction
    # --------------------------------------------------------

    probabilities = model.predict(embedding)[0]

    # --------------------------------------------------------
    # Get top-k predictions
    # --------------------------------------------------------

    top_indices = np.argsort(
        probabilities
    )[::-1][:top_k]

    top_predictions = []

    for class_id in top_indices:

        class_id = int(class_id)

        top_predictions.append({
            "label": id2label[str(class_id)],
            "confidence": float(
                probabilities[class_id]
            )
        })

    # --------------------------------------------------------
    # Final prediction
    # --------------------------------------------------------

    predicted_class = int(
        np.argmax(probabilities)
    )

    predicted_label = id2label[
        str(predicted_class)
    ]

    confidence = float(
        probabilities[predicted_class]
    )

    # --------------------------------------------------------
    # Return results
    # --------------------------------------------------------

    return {
        "text": text,
        "predicted_label": predicted_label,
        "confidence": confidence,
        "top_predictions": top_predictions
    }


print("=" * 60)
print("INFERENCE FUNCTION CREATED")
print("=" * 60)

print("\nFunction:")
print("predict_next_slot(context, top_k=3)")

print("\nReady for unseen conversations.")

INFERENCE FUNCTION CREATED

Function:
predict_next_slot(context, top_k=3)

Ready for unseen conversations.


In [4]:
# ============================================================
# CELL 4 — TEST UNSEEN CONVERSATIONS
# ============================================================

test_cases = [
    {
        "name": "Case 1",
        "expected": "pain_location",
        "context": [
            {
                "role": "user",
                "content": "I've been having a dull stomach ache since this morning."
            }
        ]
    },

    {
        "name": "Case 2",
        "expected": "pain_quality",
        "context": [
            {
                "role": "user",
                "content": "The pain is mostly in my lower abdomen."
            },
            {
                "role": "assistant",
                "content": "Can you describe what the pain feels like?"
            },
            {
                "role": "user",
                "content": "It's more of a burning sensation."
            }
        ]
    },

    {
        "name": "Case 3",
        "expected": "pain_severity",
        "context": [
            {
                "role": "user",
                "content": "The pain is on the right side of my stomach."
            },
            {
                "role": "assistant",
                "content": "How would you describe the pain?"
            },
            {
                "role": "user",
                "content": "It's a sharp pain."
            },
            {
                "role": "assistant",
                "content": "How severe is it?"
            },
            {
                "role": "user",
                "content": "I'd say about 7 out of 10."
            }
        ]
    },

    {
        "name": "Case 4",
        "expected": "pain_progression",
        "context": [
            {
                "role": "user",
                "content": "I've had abdominal pain for two days."
            },
            {
                "role": "assistant",
                "content": "Where is the pain?"
            },
            {
                "role": "user",
                "content": "Around my upper abdomen."
            },
            {
                "role": "assistant",
                "content": "How would you describe it?"
            },
            {
                "role": "user",
                "content": "It's a cramping pain."
            },
            {
                "role": "assistant",
                "content": "Has the pain changed since it started?"
            },
            {
                "role": "user",
                "content": "It seems to be getting worse."
            }
        ]
    },

    {
        "name": "Case 5",
        "expected": "cough_character",
        "context": [
            {
                "role": "user",
                "content": "I've been coughing for the past few days."
            }
        ]
    },

    {
        "name": "Case 6",
        "expected": "dizziness_onset",
        "context": [
            {
                "role": "user",
                "content": "I've suddenly started feeling dizzy today."
            }
        ]
    },

    {
        "name": "Case 7",
        "expected": "fever_temperature",
        "context": [
            {
                "role": "user",
                "content": "I've had a fever since last night."
            },
            {
                "role": "assistant",
                "content": "Do you know what your temperature has been?"
            },
            {
                "role": "user",
                "content": "It was around 102 degrees."
            }
        ]
    },

    {
        "name": "Case 8",
        "expected": "urinary_frequency",
        "context": [
            {
                "role": "user",
                "content": "I've been needing to urinate much more often than usual."
            }
        ]
    },

    {
        "name": "Case 9",
        "expected": "rash_location",
        "context": [
            {
                "role": "user",
                "content": "I noticed a new rash on my skin."
            }
        ]
    },

    {
        "name": "Case 10",
        "expected": "RED_FLAG",
        "context": [
            {
                "role": "user",
                "content": "I'm having severe chest pain and I'm struggling to breathe."
            }
        ]
    }
]


# ------------------------------------------------------------
# Run predictions
# ------------------------------------------------------------

results = []

for case in test_cases:

    prediction = predict_next_slot(
        case["context"],
        top_k=3
    )

    results.append({
        "case": case["name"],
        "expected": case["expected"],
        "predicted": prediction["predicted_label"],
        "confidence": prediction["confidence"],
        "top_3": prediction["top_predictions"]
    })


# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("=" * 60)
print("UNSEEN TEST RESULTS")
print("=" * 60)

for result in results:

    print(f"\n{result['case']}")
    print(f"Expected   : {result['expected']}")
    print(f"Predicted  : {result['predicted']}")
    print(f"Confidence : {result['confidence']:.4f}")

    print("Top 3:")
    for item in result["top_3"]:
        print(
            f"  {item['label']:30s} "
            f"{item['confidence']:.4f}"
        )

UNSEEN TEST RESULTS

Case 1
Expected   : pain_location
Predicted  : pain_location
Confidence : 0.9882
Top 3:
  pain_location                  0.9882
  fever_temperature              0.0064
  pain_severity                  0.0005

Case 2
Expected   : pain_quality
Predicted  : pain_severity
Confidence : 0.1716
Top 3:
  pain_severity                  0.1716
  associated_symptoms            0.1071
  urinary_frequency              0.0490

Case 3
Expected   : pain_severity
Predicted  : pain_progression
Confidence : 0.6762
Top 3:
  pain_progression               0.6762
  associated_symptoms            0.0969
  RED_FLAG                       0.0373

Case 4
Expected   : pain_progression
Predicted  : functional_impact
Confidence : 0.8493
Top 3:
  functional_impact              0.8493
  associated_symptoms            0.1244
  pain_relieving_factors         0.0026

Case 5
Expected   : cough_character
Predicted  : cough_character
Confidence : 0.9979
Top 3:
  cough_character                0.9979
  

In [6]:
# ============================================================
# MODEL 2 — COMPLETE DATASET FORENSIC AUDIT
# ============================================================

import os
import pandas as pd
from collections import Counter, defaultdict
from difflib import SequenceMatcher

print("=" * 80)
print("MODEL 2 — COMPLETE DATASET FORENSIC AUDIT")
print("=" * 80)


# ============================================================
# 1. LOAD DATASET
# ============================================================

AUDIT_DATASET_PATH = os.path.join(
    BASE_DIR,
    "model2_preprocessed_lightgbm",
    "dataset.csv"
)

df = pd.read_csv(
    AUDIT_DATASET_PATH
)

print("\nDataset loaded successfully.")
print("Path:", AUDIT_DATASET_PATH)


# ============================================================
# 2. BASIC DATASET INFORMATION
# ============================================================

print("\n" + "=" * 80)
print("1. BASIC DATASET INFORMATION")
print("=" * 80)

print(f"Total records  : {len(df)}")
print(f"Total labels   : {df['label'].nunique()}")
print(f"RED_FLAG       : {(df['label'] == 'RED_FLAG').sum()}")
print(f"Normal records : {(df['label'] != 'RED_FLAG').sum()}")


# ============================================================
# 3. LABEL DISTRIBUTION
# ============================================================

print("\n" + "=" * 80)
print("2. LABEL DISTRIBUTION")
print("=" * 80)

label_counts = (
    df["label"]
    .value_counts()
    .sort_values(ascending=False)
)

for label, count in label_counts.items():

    percentage = (count / len(df)) * 100

    print(
        f"{label:30s} | "
        f"{count:3d} examples | "
        f"{percentage:6.2f}%"
    )


# ============================================================
# 4. ALL EXAMPLES GROUPED BY LABEL
# ============================================================

print("\n" + "=" * 80)
print("3. ALL CONVERSATIONS GROUPED BY LABEL")
print("=" * 80)

for label in sorted(df["label"].unique()):

    label_df = df[df["label"] == label]

    print("\n" + "-" * 80)
    print(f"LABEL: {label}")
    print(f"EXAMPLES: {len(label_df)}")
    print("-" * 80)

    for _, row in label_df.iterrows():

        print(f"\nID: {row['id']}")
        print(f"Text: {row['text']}")
        print(f"Target: {row['target']}")
        print(f"Red Flag: {row['red_flag']}")


# ============================================================
# 5. EXACT DUPLICATE TEXT WITH DIFFERENT LABELS
# ============================================================

print("\n" + "=" * 80)
print("4. EXACT DUPLICATE TEXT CHECK")
print("=" * 80)

text_groups = defaultdict(list)

for _, row in df.iterrows():

    text_groups[row["text"]].append(
        (row["id"], row["label"])
    )

conflicting_duplicates = []

for text, items in text_groups.items():

    labels = set(
        label for _, label in items
    )

    if len(labels) > 1:

        conflicting_duplicates.append(
            (text, items)
        )

if not conflicting_duplicates:

    print(
        "No exact duplicate texts with different labels found."
    )

else:

    print(
        f"Found {len(conflicting_duplicates)} "
        "conflicting duplicate groups."
    )

    for text, items in conflicting_duplicates:

        print("\nTEXT:")
        print(text)

        for item in items:
            print(
                f"  {item[0]} -> {item[1]}"
            )


# ============================================================
# 6. CONVERSATION LENGTH ANALYSIS
# ============================================================

print("\n" + "=" * 80)
print("5. CONVERSATION LENGTH ANALYSIS")
print("=" * 80)

length_data = []

for _, row in df.iterrows():

    text = row["text"]

    length_data.append({
        "id": row["id"],
        "label": row["label"],
        "user_messages": text.count("User:"),
        "assistant_messages": text.count("Assistant:"),
        "characters": len(text),
        "words": len(text.split())
    })

length_df = pd.DataFrame(length_data)

print("\nOverall statistics:")
print(
    length_df[
        [
            "user_messages",
            "assistant_messages",
            "characters",
            "words"
        ]
    ].describe().round(2).to_string()
)


# ============================================================
# 7. VERY SHORT EXAMPLES
# ============================================================

print("\n" + "=" * 80)
print("6. VERY SHORT EXAMPLES")
print("=" * 80)

short_examples = length_df[
    length_df["words"] <= 8
]

print(
    f"Examples with <= 8 words: "
    f"{len(short_examples)}"
)

for _, row in short_examples.iterrows():

    original = df[
        df["id"] == row["id"]
    ].iloc[0]

    print("\nID:", row["id"])
    print("Label:", row["label"])
    print("Words:", row["words"])
    print("Text:", original["text"])


# ============================================================
# 8. HIGH-SIMILARITY TEXTS WITH DIFFERENT LABELS
# ============================================================

print("\n" + "=" * 80)
print("7. HIGH-SIMILARITY EXAMPLES WITH DIFFERENT LABELS")
print("=" * 80)

similar_pairs = []

for i in range(len(df)):

    for j in range(i + 1, len(df)):

        label_i = df.iloc[i]["label"]
        label_j = df.iloc[j]["label"]

        # Only compare different labels
        if label_i == label_j:
            continue

        text_i = str(df.iloc[i]["text"]).lower()
        text_j = str(df.iloc[j]["text"]).lower()

        similarity = SequenceMatcher(
            None,
            text_i,
            text_j
        ).ratio()

        if similarity >= 0.65:

            similar_pairs.append({
                "id_1": df.iloc[i]["id"],
                "label_1": label_i,
                "id_2": df.iloc[j]["id"],
                "label_2": label_j,
                "similarity": similarity,
                "text_1": df.iloc[i]["text"],
                "text_2": df.iloc[j]["text"]
            })


similar_pairs.sort(
    key=lambda x: x["similarity"],
    reverse=True
)

print(
    f"Found {len(similar_pairs)} "
    "high-similarity pairs."
)

for pair in similar_pairs[:30]:

    print("\n" + "-" * 80)

    print(
        f"{pair['id_1']} [{pair['label_1']}]"
        f"  <->  "
        f"{pair['id_2']} [{pair['label_2']}]"
    )

    print(
        f"Similarity: "
        f"{pair['similarity']:.3f}"
    )

    print("\nText 1:")
    print(pair["text_1"])

    print("\nText 2:")
    print(pair["text_2"])


# ============================================================
# 9. RARE LABELS
# ============================================================

print("\n" + "=" * 80)
print("8. EXTREMELY RARE LABELS")
print("=" * 80)

rare_labels = label_counts[
    label_counts <= 2
]

print(
    f"Labels with <= 2 examples: "
    f"{len(rare_labels)}"
)

for label, count in rare_labels.items():

    print(
        f"{label:30s}: {count}"
    )


# ============================================================
# 10. RED FLAG AUDIT
# ============================================================

print("\n" + "=" * 80)
print("9. RED FLAG AUDIT")
print("=" * 80)

red_flag_df = df[
    df["label"] == "RED_FLAG"
]

print(
    f"Total RED_FLAG examples: "
    f"{len(red_flag_df)}"
)

for _, row in red_flag_df.iterrows():

    print("\nID:", row["id"])
    print("Text:", row["text"])


# ============================================================
# 11. CLINICAL LABEL GROUPS
# ============================================================

print("\n" + "=" * 80)
print("10. RELATED LABEL GROUPS")
print("=" * 80)

related_groups = {

    "PAIN": [
        "pain_location",
        "pain_quality",
        "pain_severity",
        "pain_progression",
        "pain_triggers",
        "pain_relieving_factors",
        "associated_symptoms",
        "functional_impact"
    ],

    "COUGH": [
        "cough_character",
        "cough_frequency",
        "cough_progression"
    ],

    "DIZZINESS": [
        "dizziness_onset",
        "dizziness_frequency",
        "dizziness_progression",
        "dizziness_triggers",
        "dizziness_type"
    ],

    "FEVER": [
        "fever_temperature",
        "fever_progression"
    ],

    "URINARY": [
        "urinary_onset",
        "urinary_frequency",
        "urinary_progression"
    ],

    "RASH": [
        "rash_location",
        "rash_character",
        "rash_progression"
    ],

    "BREATHING": [
        "breath_onset",
        "breath_severity",
        "breath_progression"
    ]
}

for group_name, labels in related_groups.items():

    print(f"\n{group_name}")

    for label in labels:

        count = int(
            (df["label"] == label).sum()
        )

        print(
            f"  {label:30s}: {count}"
        )


# ============================================================
# 12. SAVE AUDIT REPORT
# ============================================================

AUDIT_PATH = os.path.join(
    PREPROCESSED_DIR,
    "dataset_audit.txt"
)

with open(
    AUDIT_PATH,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        "MODEL 2 — DATASET FORENSIC AUDIT\n"
    )

    f.write("=" * 80 + "\n\n")

    f.write(
        f"Total records: {len(df)}\n"
    )

    f.write(
        f"Total labels: {df['label'].nunique()}\n\n"
    )

    f.write("LABEL DISTRIBUTION\n")
    f.write("-" * 80 + "\n")

    for label, count in label_counts.items():

        percentage = (
            count / len(df)
        ) * 100

        f.write(
            f"{label:30s} | "
            f"{count:3d} | "
            f"{percentage:6.2f}%\n"
        )

    f.write("\n\nALL EXAMPLES\n")
    f.write("=" * 80 + "\n")

    for label in sorted(df["label"].unique()):

        f.write(
            f"\n\nLABEL: {label}\n"
        )

        f.write("-" * 80 + "\n")

        for _, row in df[
            df["label"] == label
        ].iterrows():

            f.write(
                f"\nID: {row['id']}\n"
            )

            f.write(
                f"Text: {row['text']}\n"
            )

            f.write(
                f"Target: {row['target']}\n"
            )

            f.write(
                f"Red Flag: {row['red_flag']}\n"
            )


# ============================================================
# FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("AUDIT COMPLETE")
print("=" * 80)

print(f"\nRecords                  : {len(df)}")
print(f"Labels                   : {df['label'].nunique()}")
print(f"RED_FLAG                 : {(df['label'] == 'RED_FLAG').sum()}")
print(f"Rare labels (<=2)        : {len(rare_labels)}")
print(f"Conflicting duplicates   : {len(conflicting_duplicates)}")
print(f"High-similarity pairs    : {len(similar_pairs)}")

print("\nAudit report saved to:")
print(AUDIT_PATH)

print("\n" + "=" * 80)
print("SEND ME THE OUTPUT")
print("=" * 80)

MODEL 2 — COMPLETE DATASET FORENSIC AUDIT

Dataset loaded successfully.
Path: c:\Users\astha\Desktop\sih2026\Modular-medical-intelligence-and-Dialogue-system\model_2(Medical Dialogue Manager)\model2_preprocessed_lightgbm\dataset.csv

1. BASIC DATASET INFORMATION
Total records  : 100
Total labels   : 36
RED_FLAG       : 15
Normal records : 85

2. LABEL DISTRIBUTION
associated_symptoms            |  25 examples |  25.00%
RED_FLAG                       |  15 examples |  15.00%
functional_impact              |   8 examples |   8.00%
pain_location                  |   7 examples |   7.00%
pain_severity                  |   6 examples |   6.00%
pain_quality                   |   5 examples |   5.00%
injury_or_trigger              |   3 examples |   3.00%
pain_progression               |   2 examples |   2.00%
cough_character                |   2 examples |   2.00%
pain_triggers                  |   1 examples |   1.00%
pain_relieving_factors         |   1 examples |   1.00%
cough_frequency  